# ASQA layer reconciliation
Row counts per table across Source (Dataverse), Cleansed and Unified.
Result lands in a DataFrame and the temp view `recon_counts` so you can run SQL on it.

In [ ]:
# ---- CONFIG ----
SOURCE_DB       = "dataverse_asqaarms01de_unqde396ad0f22d448c94c779e588420"
SILVER_DB       = "silver2_db"      # cleansed and unified both live here

CLEANSED_PREFIX = "cleansed_arms_"  # cleansed_arms_account style
UNIFIED_PREFIX  = ""                # change if unified tables carry a prefix

# (source table, cleansed table, unified table) - same as the register
TABLES = [
    ("asqa_asqainvoice", "asqa_asqainvoice", "invoice"),
    ("asqa_elicoscourse", "asqa_elicoscourse", "master_elicos_course"),
    ("asqa_feesandchanges", "asqa_feesandchanges", "master_fees_and_charges"),
    ("asqa_governingpersonrole", "asqa_governingpersonrole", "master_governing_person_role"),
    ("asqa_registrationsubmission", "asqa_registrationsubmission", "master_registration_submission"),
    ("asqa_registrationsubmission", "asqa_registrationsubmission", "registration_submission"),
    ("asqa_registrationsubmissiontype", "asqa_registrationsubmissiontype", "master_registration_submission_type"),
    ("asqa_rfi", "asqa_rfi", "reg_sub_rfi"),
    ("asqa_rmasignificantevent", "asqa_rmasignificantevent", "reg_sub_rma_significant_event"),
    ("asqa_rmhistory", "asqa_rmhistory", "reg_sub_rm_history"),
    ("asqa_rtoclosure", "asqa_rtoclosure", "reg_sub_rto_closure"),
    ("asqa_rtodelegation", "asqa_rtodelegation", "reg_sub_rto_delegation"),
    ("asqa_rtoevidenceofdelivery", "asqa_rtoevidenceofdelivery", "reg_sub_rto_evidence_of_delivery"),
    ("asqa_rtoextendmeasurementperiod", "asqa_rtoextendmeasurementperiod", "reg_sub_rto_extend_measurement_period"),
    ("asqa_rtofinancialstatus1", "asqa_rtofinancialstatus1", "reg_sub_rto_financial_status"),
    ("asqa_rtofitandproperperson", "asqa_rtofitandproperperson", "reg_sub_rto_fit_and_proper_person"),
    ("asqa_rtolapsing", "asqa_rtolapsing", "reg_sub_rto_lapsing"),
    ("asqa_rtolegaldetails", "asqa_rtolegaldetails", "reg_sub_rto_legal_detail"),
    ("asqa_rtoowner", "asqa_rtoowner", "reg_sub_rto_owner"),
    ("asqa_rtoownership", "asqa_rtoownership", "reg_sub_rto_ownership"),
    ("asqa_rtopublishrequest", "asqa_rtopublishrequest", "rto_publish_request"),
    ("asqa_rtorenewal", "asqa_rtorenewal", "reg_sub_rto_renewal"),
    ("asqa_rtotypeofsignificantevent", "asqa_rtotypeofsignificantevent", "master_rto_type_of_significant_event"),
    ("asqa_rtowithdrawal", "asqa_rtowithdrawal", "reg_sub_rto_withdrawal"),
    ("asqa_studentmanagementprovider", "asqa_studentmanagementprovider", "master_student_management_provider"),
    ("asqa_studentmanagementsystem", "asqa_studentmanagementsystem", "reg_sub_student_management_system"),
    ("asqa_thirdpartyarrangements", "asqa_thirdpartyarrangements", "reg_sub_third_party_arrangement"),
]

In [ ]:
# ---- COUNT HELPER ----
def count_table(db, table):
    """Returns (count, comment). count is None if the table could not be read."""
    full = f"`{db}`.`{table}`"
    try:
        n = spark.sql(f"SELECT COUNT(*) AS c FROM {full}").collect()[0]["c"]
        return n, None
    except Exception as e:
        msg = str(e)
        low = msg.lower()
        if "not found" in low or "cannot be found" in low or "does not exist" in low or "table_or_view_not_found" in low:
            return None, "table not found"
        return None, "error: " + msg.strip().split("\n")[0][:200]

In [ ]:
# ---- RUN ----
from pyspark.sql.types import StructType, StructField, StringType, LongType

results = []
for src_t, cln_t, uni_t in TABLES:
    cln_full = CLEANSED_PREFIX + cln_t
    uni_full = UNIFIED_PREFIX + uni_t

    s_cnt, s_err = count_table(SOURCE_DB, src_t)
    c_cnt, c_err = count_table(SILVER_DB, cln_full)
    u_cnt, u_err = count_table(SILVER_DB, uni_full)

    comments = []
    if s_err: comments.append(f"source: {s_err}")
    if c_err: comments.append(f"cleansed: {c_err}")
    if u_err: comments.append(f"unified: {u_err}")

    results.append((src_t, s_cnt, cln_full, c_cnt, uni_full, u_cnt, "; ".join(comments) or None))

schema = StructType([
    StructField("source_table",   StringType(), True),
    StructField("source_count",   LongType(),   True),
    StructField("cleansed_table", StringType(), True),
    StructField("cleansed_count", LongType(),   True),
    StructField("unified_table",  StringType(), True),
    StructField("unified_count",  LongType(),   True),
    StructField("comment",        StringType(), True),
])

recon_df = spark.createDataFrame(results, schema)
recon_df.createOrReplaceTempView("recon_counts")
display(recon_df)

In [ ]:
%%sql
-- examples: run your own against recon_counts
SELECT *
FROM recon_counts
WHERE comment IS NOT NULL
   OR source_count   <> cleansed_count
   OR cleansed_count <> unified_count